In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
!pip install --upgrade "scikit-learn>=1.5.0,<1.6.0" "scikit-survival>=0.23.0" ucimlrepo

INFO: pip is looking at multiple versions of scikit-survival to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.0/49.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 113.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 73.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.1/222.1 kB 12.9 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tpo

<h1>the DRM model </h1>

In [3]:
"""
Deep Cox Mixtures (Nagpal et al., 2021) -- Algorithm 1, Section 3.4.
Expanded to support Generalized Odds-Rate (proportional rho-odds) components and
parametric (Weibull) baselines.

Per-component model (Scharfstein, Tsiatis & Gilbert, 1998):
    S_k(t|x)   = (1 + rho * A_k(t) * exp(f_k(x)))^(-1/rho)       (rho = 0 -> exp(-A_k e^f))
    lambda_k   = a_k(t) exp(f_k) / (1 + rho * A_k(t) * exp(f_k))
    L(theta)   = prod_i sum_k pi_k(x_i) * lambda_k(t_i|x_i)^delta_i * S_k(t_i|x_i)

alpha_mode:
  "parametric": A_k(t) = exp(c_k) t^gamma_k, trained by maximising the full
                complete-data likelihood -> valid for any rho >= 0.
  "spline":     A_k(t) from a Breslow estimate + spline (as in the DCM paper).
                The network is trained with the Cox PARTIAL likelihood, which is only
                valid for rho = 0. For rho > 0 it is an approximation (rho only enters
                the E-step / prediction, not the M-step) -- use "parametric" instead.

rho:
  * fixed (default):      rho is a hyperparameter, passed as `rho=...`.
  * learnable:            `learn_rho=True` (parametric mode only). rho is parametrised as
                          softplus(raw_rho) > 0, initialised at `rho`, and optimised jointly
                          with the network and the Weibull baselines on the complete-data
                          likelihood. `rho_per_component=True` gives each of the K components
                          its own rho_k instead of one shared rho.
  rho = 0 (Cox) is the boundary of the parameter space and is reached only in the limit;
  the likelihood is evaluated with a series expansion for rho * A * e^f -> 0 so it stays
  numerically stable (and differentiable w.r.t. rho) there.
  Learning rho needs the full likelihood, so it is rejected for alpha_mode="spline".

Differences from the paper's Algorithm 1: no VAE regulariser (alpha = 0), and the
Breslow/spline baselines are refit once per epoch by default (see `refit_every`).
"""
import math
import warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.interpolate import UnivariateSpline
from scipy.ndimage import gaussian_filter1d


def _inv_softplus(x):
    """Inverse of softplus for x > 0: log(exp(x) - 1), computed stably."""
    return x + math.log(-math.expm1(-x))


def _gor_log_terms(z, rho):
    """
    z = A(t) * exp(f(x)) >= 0, rho >= 0 (0-dim tensor, may require grad).
    Returns (log S, log(1 + rho*z)) where log S = -(1/rho) * log(1 + rho*z).

    For u = rho*z < 1e-3 we use log1p(u)/u = 1 - u/2 + u^2/3 - u^3/4 + ..., which avoids
    the 0/0 at rho = 0 and the catastrophic cancellation in d/d(rho) for small rho.
    """
    u = rho * z
    small = u < 1e-3
    u_safe = torch.where(small, torch.ones_like(u), u)   # keeps the log branch finite
    u_ser = torch.clamp(u, max=1e-3)                     # keeps the series branch finite
    g_series = 1.0 - u_ser / 2.0 + u_ser ** 2 / 3.0 - u_ser ** 3 / 4.0
    g = torch.where(small, g_series, torch.log1p(u_safe) / u_safe)
    return -z * g, torch.log1p(u)


def _gor_log_surv_np(z, rho):
    """NumPy version of the log-survival from `_gor_log_terms` (prediction only)."""
    u = rho * z
    small = u < 1e-3
    u_safe = np.where(small, 1.0, u)
    u_ser = np.minimum(u, 1e-3)
    g = np.where(small, 1.0 - u_ser / 2.0 + u_ser ** 2 / 3.0 - u_ser ** 3 / 4.0,
                 np.log1p(u_safe) / u_safe)
    return -z * g


class DCMNet(nn.Module):
    def __init__(self, in_dim, k=3, hidden=100, n_layers=1, alpha_mode="spline",
                 rho=0.0, learn_rho=False, rho_per_component=False):
        super().__init__()
        layers, d = [], in_dim
        for _ in range(n_layers):
            layers += [nn.Linear(d, hidden), nn.ReLU()]
            d = hidden
        self.encoder = nn.Sequential(*layers)
        self.gate = nn.Linear(d, k)
        self.hazard = nn.Linear(d, k)
        self.k = k
        self.alpha_mode = alpha_mode

        # Learnable baseline parameters for parametric alpha(T) transformation
        if self.alpha_mode == "parametric":
            self.log_gamma = nn.Parameter(torch.zeros(k))
            self.c = nn.Parameter(torch.zeros(k))

        # rho: learnable (softplus-parametrised, so rho > 0) or a fixed buffer.
        # Living inside the module means it is saved/restored with state_dict(),
        # so early stopping restores the best rho together with the best weights.
        self.learn_rho = learn_rho
        n_rho = k if rho_per_component else 1
        if learn_rho:
            self.raw_rho = nn.Parameter(torch.full((n_rho,), _inv_softplus(float(rho))))
        else:
            self.register_buffer("rho_fixed", torch.full((n_rho,), float(rho)))

    def get_rho(self):
        """rho for every component, shape (k,). Differentiable when learn_rho=True."""
        r = F.softplus(self.raw_rho) if self.learn_rho else self.rho_fixed
        return r.expand(self.k)

    def forward(self, x):
        z = self.encoder(x)
        return F.log_softmax(self.gate(z), dim=-1), self.hazard(z)


class BaselineSpline:
    def __init__(self, t_evt, Lam_evt, n_grid=200, sigma=2.0):
        t_hi = float(t_evt.max())
        g = np.linspace(0.0, t_hi, n_grid)
        if t_evt[0] > 0:
            xp, fp = np.r_[0.0, t_evt], np.r_[0.0, Lam_evt]
        else:
            xp, fp = t_evt, Lam_evt
        Lg = np.interp(g, xp, fp)
        # Smooth with point-reflection padding (odd extension about the end points):
        # keeps Lambda(0) = 0 and does not flatten the upper end, unlike mode="nearest".
        pad = int(np.ceil(4 * sigma))
        if sigma > 0 and len(Lg) > pad + 1:
            left = 2 * Lg[0] - Lg[pad:0:-1]
            right = 2 * Lg[-1] - Lg[-2:-pad - 2:-1]
            Lg = gaussian_filter1d(np.r_[left, Lg, right], sigma, mode="nearest")[pad:-pad]
        Lg = np.maximum.accumulate(np.maximum(Lg, 0.0))
        self.spl = UnivariateSpline(g, Lg, k=3, s=0)
        self.dspl = self.spl.derivative()
        self.t_max = t_hi
        # hazard / cumulative hazard at t_max, used for a constant-hazard tail
        self.lam_end = float(max(self.dspl(t_hi), 1e-8))
        self.Lam_end = float(max(self.spl(t_hi), 0.0))

    def Lam(self, t):
        t = np.asarray(t, float)
        inside = np.maximum(self.spl(np.clip(t, 0.0, self.t_max)), 0.0)
        # beyond the last event time: constant-hazard extrapolation, so that Lam and
        # dS stay consistent (previously Lam was flat while the hazard was non-zero)
        return np.where(t > self.t_max, self.Lam_end + self.lam_end * (t - self.t_max), inside)

    def S(self, t):
        return np.clip(np.exp(-self.Lam(t)), 1e-6, 1.0)

    def dS(self, t):
        lam = np.maximum(self.dspl(np.clip(t, 0.0, self.t_max)), 1e-8)
        return -lam * self.S(t)  # beyond t_max: hazard = hazard at t_max (see Lam)


def breslow_baseline(t, delta, f, mask, min_events=5):
    idx = np.where(mask)[0]
    if len(idx) == 0 or delta[idx].sum() < min_events:
        return None
    tt, dd, ff = t[idx], delta[idx], f[idx]
    order = np.argsort(tt, kind="stable")
    tt, dd, ff = tt[order], dd[order], ff[order]
    log_risk = np.logaddexp.accumulate(ff[::-1])[::-1]
    ut, first, inv = np.unique(tt, return_index=True, return_inverse=True)
    d_u = np.bincount(inv, weights=dd, minlength=len(ut))
    jump = d_u * np.exp(-log_risk[first])
    Lam = np.cumsum(jump)
    keep = d_u > 0
    if keep.sum() < min_events:
        return None
    return ut[keep], Lam[keep]


def cox_neg_partial_ll(f, t, delta):
    if delta.sum() == 0:
        return f.sum() * 0.0
    order = torch.argsort(t, descending=True)
    f, t, delta = f[order], t[order], delta[order]
    log_cum = torch.logcumsumexp(f, dim=0)
    _, inv, counts = torch.unique_consecutive(t, return_inverse=True,
                                              return_counts=True)
    last = torch.cumsum(counts, dim=0) - 1
    log_risk = log_cum[last][inv]
    return -((f - log_risk) * delta).sum()


class DeepCoxMixture:
    def __init__(self, in_dim, k=3, hidden=100, n_layers=1,
                 lr=1e-3, batch_size=128, refit_every=None, min_events=5,
                 device=None, seed=0, rho=None, alpha_mode="spline",
                 learn_rho=False, rho_per_component=False):
        """
        rho:               fixed value if learn_rho=False (default 0.0 = Cox); initial value
                           if learn_rho=True (default 1.0, must be > 0).
        learn_rho:         learn rho by gradient ascent on the complete-data likelihood
                           (alpha_mode="parametric" only).
        rho_per_component: one rho_k per mixture component instead of a single shared rho
                           (only matters when learn_rho=True; a fixed rho is shared anyway).
        """
        if alpha_mode not in ("spline", "parametric"):
            raise ValueError("alpha_mode must be 'spline' or 'parametric'")
        if rho is None:
            rho = 1.0 if learn_rho else 0.0
        if rho < 0:
            raise ValueError("rho must be >= 0 (rho=0: Cox, rho=1: proportional odds)")
        if learn_rho and alpha_mode != "parametric":
            raise ValueError("learn_rho=True requires alpha_mode='parametric': with the "
                             "spline/Breslow baseline the network is trained on the Cox partial "
                             "likelihood, which does not depend on rho, so rho cannot be learned.")
        if learn_rho and rho == 0:
            raise ValueError("learn_rho=True needs an initial rho > 0 (rho is parametrised as "
                             "softplus(raw_rho) > 0); rho = 0 is reached only in the limit.")
        if alpha_mode == "spline" and rho > 0:
            warnings.warn("alpha_mode='spline' with rho>0: the network is trained with the Cox "
                          "partial likelihood and a Breslow baseline, so rho does not enter the "
                          "M-step (not an MLE of the rho-odds model). Use alpha_mode='parametric' "
                          "for rho>0.", UserWarning)
        torch.manual_seed(seed)
        np.random.seed(seed)
        self.k = k
        self.bs = batch_size
        self.refit_every = refit_every
        self.min_events = min_events
        self.learn_rho = learn_rho
        self.alpha_mode = alpha_mode
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.net = DCMNet(in_dim, k, hidden, n_layers, alpha_mode,
                          rho=rho, learn_rho=learn_rho,
                          rho_per_component=rho_per_component).to(self.device)
        self.opt = torch.optim.Adam(self.net.parameters(), lr=lr)

        self.splines = [None] * k
        self.fallback = [False] * k

    @property
    def rho(self):
        """Current rho of each component as a NumPy array of shape (k,)."""
        with torch.no_grad():
            return self.net.get_rho().cpu().numpy()

    @torch.no_grad()
    def _forward_all(self, X):
        self.net.eval()
        logpi, f = self.net(X)
        self.net.train()
        return logpi, f

    def _refit_baselines(self, t_np, d_np, f_np, zeta_np):
        everyone = np.ones(len(t_np), dtype=bool)
        for j in range(self.k):
            out = breslow_baseline(t_np, d_np, f_np[:, j], zeta_np == j,
                                   self.min_events)
            self.fallback[j] = out is None
            if out is None:
                out = breslow_baseline(t_np, d_np, f_np[:, j], everyone,
                                       self.min_events)
            if out is None:
                raise RuntimeError("Not enough events to fit any baseline.")
            self.splines[j] = BaselineSpline(*out)

    def _get_log_likelihoods(self, t_b, d_b, f_tensor):
        """Unified likelihood computer for EM steps."""
        tb = torch.tensor(t_b, dtype=torch.float32, device=self.device)
        db = torch.tensor(d_b, dtype=torch.float32, device=self.device)
        logp = torch.zeros((len(t_b), self.k), device=self.device)

        if self.alpha_mode == "spline":
            # rho is fixed in spline mode (learn_rho is rejected in __init__)
            rho_np = self.rho
            f_np = f_tensor.detach().cpu().numpy()
            for j in range(self.k):
                sp = self.splines[j]
                if sp is None:
                    continue
                rho_j = float(rho_np[j])

                S0 = np.clip(sp.S(t_b), 1e-12, 1.0)
                dS0 = sp.dS(t_b)
                A0 = -np.log(S0)
                a0 = -dS0 / S0
                ef = np.exp(np.clip(f_np[:, j], -20, 20))

                if rho_j > 0:
                    S = (1 + rho_j * A0 * ef) ** (-1 / rho_j)
                    dens = a0 * ef * (1 + rho_j * A0 * ef) ** (-(1 / rho_j + 1))
                else:
                    S = S0 ** ef
                    dens = a0 * ef * S

                logS = np.log(np.clip(S, 1e-12, None))
                logd = np.log(np.clip(dens, 1e-12, None))
                logp[:, j] = torch.tensor(np.where(d_b == 1, logd, logS),
                                          dtype=torch.float32, device=self.device)
        else:
            shape = torch.exp(self.net.log_gamma)   # Weibull shape (not the E-step responsibilities)
            c = self.net.c
            rho = self.net.get_rho()                # (k,); carries grad when learn_rho=True
            tb_clamp = torch.clamp(tb, 1e-6)
            for j in range(self.k):
                A_j = torch.exp(c[j]) * (tb_clamp ** shape[j])
                a_j = shape[j] * torch.exp(c[j]) * (tb_clamp ** (shape[j] - 1.0))
                ef_j = torch.exp(torch.clamp(f_tensor[:, j], -20, 20))

                # log S = -(1/rho) log(1 + rho A e^f);  log lambda = log a + f - log(1 + rho A e^f)
                # (rho = 0 gives log S = -A e^f, log lambda = log a + f)
                log_S, log1p_u = _gor_log_terms(A_j * ef_j, rho[j])
                log_dens = torch.log(a_j + 1e-10) + f_tensor[:, j] - log1p_u + log_S

                logp[:, j] = db * log_dens + (1 - db) * log_S

        return logp

    @torch.no_grad()
    def _e_step(self, t_b, d_b, logpi_b, f_tensor):
        # no_grad: responsibilities must be treated as constants in the M-step. Without it,
        # in parametric mode the gate term leaks gradients into log_gamma / c / rho.
        logp = self._get_log_likelihoods(t_b, d_b, f_tensor.detach())
        gamma = F.softmax(logpi_b.detach() + logp, dim=1)
        zeta = torch.distributions.Categorical(probs=gamma).sample()
        return gamma, zeta

    @torch.no_grad()
    def _full_refit(self, Xt, t_np, d_np, zeta_all):
        logpi, f = self._forward_all(Xt)
        logp = self._get_log_likelihoods(t_np, d_np, f.detach())
        gamma = F.softmax(logpi.detach() + logp, dim=1)
        zeta_all[:] = torch.distributions.Categorical(probs=gamma).sample().cpu().numpy()

        if self.alpha_mode == "spline":
            self._refit_baselines(t_np, d_np, f.cpu().numpy(), zeta_all)

    def fit(self, X, t, delta, epochs=50, X_val=None, t_val=None, d_val=None,
            verbose=True, patience=None, eval_every=5):
        import copy

        Xt = torch.tensor(X, dtype=torch.float32, device=self.device)
        tt = torch.tensor(t, dtype=torch.float32, device=self.device)
        dt = torch.tensor(delta, dtype=torch.float32, device=self.device)
        t_np, d_np = np.asarray(t, float), np.asarray(delta, int)
        N = len(t_np)

        zeta_all = np.random.randint(0, self.k, N)
        _, f0 = self._forward_all(Xt)

        if self.alpha_mode == "spline":
            self._refit_baselines(t_np, d_np, f0.cpu().numpy(), zeta_all)

        best_val = float("inf")
        best_state = None
        best_splines = None
        no_improve = 0

        step = 0
        for ep in range(epochs):
            perm = np.random.permutation(N)
            tot = 0.0
            for s in range(0, N, self.bs):
                b = perm[s:s + self.bs]
                xb, tb, db = Xt[b], tt[b], dt[b]
                logpi, f = self.net(xb)

                gamma, zeta = self._e_step(t_np[b], d_np[b], logpi, f)
                zeta_all[b] = zeta.cpu().numpy()

                gate_term = (gamma * logpi).sum()

                # Dynamic M-step based on alpha formulation
                if self.alpha_mode == "spline":
                    pl_term = 0.0
                    for j in range(self.k):
                        m = zeta == j
                        if m.sum() > 1:
                            pl_term = pl_term + (-cox_neg_partial_ll(f[m, j], tb[m], db[m]))
                else:
                    # In parametric mode, we maximize the explicit complete-data log-likelihood
                    # (rho, if learnable, receives its gradient through this term)
                    logp = self._get_log_likelihoods(t_np[b], d_np[b], f)
                    mask = F.one_hot(zeta, num_classes=self.k).float()
                    pl_term = (mask * logp).sum()

                loss = -(gate_term + pl_term) / len(b)

                self.opt.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(self.net.parameters(), 5.0)
                self.opt.step()
                tot += loss.item() * len(b)

                step += 1
                if self.refit_every and step % self.refit_every == 0:
                    self._full_refit(Xt, t_np, d_np, zeta_all)

            if not self.refit_every:
                self._full_refit(Xt, t_np, d_np, zeta_all)

            do_log = ep % eval_every == 0 or ep == epochs - 1
            val_nll = None
            if X_val is not None and do_log:
                val_nll = self.nll(X_val, t_val, d_val)

                if val_nll < best_val:
                    best_val = val_nll
                    best_state = copy.deepcopy(self.net.state_dict())  # includes raw_rho
                    if self.alpha_mode == "spline":
                        best_splines = copy.deepcopy(self.splines)
                    no_improve = 0
                else:
                    no_improve += 1

            if verbose and do_log:
                msg = f"epoch {ep:3d}  train Q-loss {tot / N:.4f}"
                if val_nll is not None:
                    msg += f"  val NLL {val_nll:.4f}"
                    if val_nll == best_val:
                        msg += "  (best)"
                sizes = np.bincount(zeta_all, minlength=self.k).tolist()
                msg += f"  group sizes {sizes}"
                if self.learn_rho:
                    msg += f"  rho {np.round(self.rho, 3).tolist()}"
                if self.alpha_mode == "spline" and any(self.fallback):
                    msg += f"  [pooled-baseline fallback: {[j for j, x in enumerate(self.fallback) if x]}]"
                print(msg)

            if patience is not None and X_val is not None and no_improve >= patience:
                if verbose:
                    print(f"early stopping at epoch {ep}: no val improvement "
                          f"for {patience} checks (best val NLL {best_val:.4f})")
                break

        if best_state is not None:
            self.net.load_state_dict(best_state)
            if self.alpha_mode == "spline":
                self.splines = best_splines
            self.best_val_nll = best_val
        return self

    @torch.no_grad()
    def predict_survival(self, X, times):
        Xt = torch.tensor(X, dtype=torch.float32, device=self.device)
        logpi, f = self._forward_all(Xt)
        pi = logpi.exp().cpu().numpy()
        ef = np.exp(np.clip(f.cpu().numpy(), -20, 20))
        rho = self.rho
        times = np.asarray(times, float)
        S = np.zeros((len(X), len(times)))

        if self.alpha_mode == "spline":
            for j in range(self.k):
                sp = self.splines[j]
                if sp is None: continue
                rho_j = float(rho[j])
                S0 = np.clip(sp.S(times), 1e-12, 1.0)
                A0 = -np.log(S0)
                if rho_j > 0:
                    S_j = (1.0 + rho_j * A0[None, :] * ef[:, [j]]) ** (-1.0 / rho_j)
                else:
                    S_j = S0[None, :] ** ef[:, [j]]
                S += pi[:, [j]] * S_j
        else:
            shape = torch.exp(self.net.log_gamma).cpu().numpy()
            c = self.net.c.cpu().numpy()
            times_clamp = np.clip(times, 1e-6, None)
            for j in range(self.k):
                A_j = np.exp(c[j]) * (times_clamp ** shape[j])
                S_j = np.exp(_gor_log_surv_np(A_j[None, :] * ef[:, [j]], float(rho[j])))
                S += pi[:, [j]] * S_j

        return S

    @torch.no_grad()
    def nll(self, X, t, delta):
        Xt = torch.tensor(X, dtype=torch.float32, device=self.device)
        logpi, f = self._forward_all(Xt)
        t_b = np.asarray(t, float)
        d_b = np.asarray(delta, int)

        # Calculate likelihoods identically to the E-step, but aggregate for evaluation
        logp = self._get_log_likelihoods(t_b, d_b, f)
        # log sum_k pi_k * lambda_k^delta * S_k, computed stably in log space
        return -torch.logsumexp(logpi + logp, dim=1).mean().item()

In [4]:
!pip install ucimlrepo

from ucimlrepo import fetch_ucirepo

# Fetch SUPPORT2
support2 = fetch_ucirepo(id=880)

X = support2.data.features
y = support2.data.targets

print(X.shape)
print(y.shape)

(9105, 42)
(9105, 3)


In [5]:
"""
Run Deep Cox Mixtures on SUPPORT2 (with Parametric and Rho-Odds variations).

Assumes you have already installed:
    pip install ucimlrepo scikit-survival
    
And that your modified DeepCoxMixture class is loaded in the environment.
"""
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sksurv.metrics import concordance_index_ipcw, brier_score
from ucimlrepo import fetch_ucirepo

# ------------------------------------------------------------------
# 1. Pull event time / indicator out of the raw CSV
# ------------------------------------------------------------------
print("Fetching SUPPORT2 dataset...")
support2 = fetch_ucirepo(id=880)
X = support2.data.features
y = support2.data.targets

raw = support2.data.original
raw_cols = {c.lower().replace(".", "_"): c for c in raw.columns}

dcol_raw = raw_cols.get("death")
tcol_raw = raw_cols.get("d_time") or raw_cols.get("dtime") or raw_cols.get("time")

if dcol_raw is None or tcol_raw is None:
    raise KeyError("Couldn't find death / d.time even in the raw CSV.")

delta = raw[dcol_raw].astype(int).values
t = raw[tcol_raw].astype(float).values

X = X.loc[raw.index]

# spline fitting needs strictly positive times
keep = t > 0
X_df = X.loc[keep].reset_index(drop=True)
t, delta = t[keep], delta[keep]

outcome_like = {"death", "hospdead", "sfdm2", "d_time", "dtime", "time"}
X_df = X_df.drop(columns=[c for c in X_df.columns
                          if c.lower().replace(".", "_") in outcome_like])

print(f"n = {len(t)}, censoring rate = {1 - delta.mean():.2%}")

# ------------------------------------------------------------------
# 2. Clean features: drop outcome-adjacent columns, encode, impute, scale
# ------------------------------------------------------------------
leak = {"hospdead", "sfdm2", "slos", "charges", "totcst", "totmcst",
        "avtisst", "surv2m", "surv6m", "prg2m", "prg6m", "dnr", "dnrday"}
drop_cols = [c for c in X_df.columns if c.lower().replace(".", "_") in leak]
X_df = X_df.drop(columns=drop_cols)

cat_cols = X_df.select_dtypes(include=["object", "category"]).columns.tolist()
X_enc = pd.get_dummies(X_df, columns=cat_cols, dummy_na=False)
X_enc = X_enc.astype(float)
X_enc = X_enc.fillna(X_enc.median())
X_enc = (X_enc - X_enc.mean()) / X_enc.std().replace(0, 1)

X_arr = X_enc.values.astype("float32")
feature_names = X_enc.columns.tolist()
print(f"Final feature matrix: {X_arr.shape}")

# ------------------------------------------------------------------
# 3. Train / val / test split
# ------------------------------------------------------------------
X_tr, X_tmp, t_tr, t_tmp, d_tr, d_tmp = train_test_split(
    X_arr, t, delta, test_size=0.4, random_state=0, stratify=delta)
X_val, X_te, t_val, t_te, d_val, d_te = train_test_split(
    X_tmp, t_tmp, d_tmp, test_size=0.5, random_state=0, stratify=d_tmp)

print(f"Train {X_tr.shape[0]}  Val {X_val.shape[0]}  Test {X_te.shape[0]}\n")

# Prepare structured arrays for scikit-survival IPCW evaluation
def make_structured_array(d, t_arr):
    return np.array(list(zip(d.astype(bool), t_arr)), 
                    dtype=[('Status', '?'), ('Survival_in_days', '<f8')])

train_y = make_structured_array(d_tr, t_tr)
test_y = make_structured_array(d_te, t_te)

# Define evaluation horizons based on training event times
horizons = np.quantile(t_tr[d_tr == 1], [0.25, 0.5, 0.75])
print(f"Evaluation Horizons (25th, 50th, 75th quantiles): {horizons}\n")

# ------------------------------------------------------------------
# 4 & 5. Fit DCM Variations & Evaluate
# ------------------------------------------------------------------
# Test configurations: Standard DCM (spline, rho=0), Parametric Cox (parametric, rho=0), Parametric Odds (parametric, rho=1)
configurations = [
    {"alpha_mode": "spline", "rho": 0.0},
    {"alpha_mode": "parametric", "rho": 0.0}, 
    {"alpha_mode": "parametric", "rho": 1.0}
]

for config in configurations:
    mode = config["alpha_mode"]
    rho_val = config["rho"]
    
    print("="*60)
    print(f"Training Model -> alpha_mode: '{mode}', rho: {rho_val}")
    print("="*60)
    
    model = DeepCoxMixture(
        in_dim=X_arr.shape[1],
        k=3,
        hidden=100,
        n_layers=1,
        lr=1e-3,
        batch_size=128,
        refit_every=None,
        min_events=15,
        seed=0,
        rho=rho_val,         # <-- NEW: Passes generalized odds-rate parameter
        alpha_mode=mode      # <-- NEW: Toggles between Spline and Parametric
    )

    model.fit(X_tr, t_tr, d_tr, epochs=100,
              X_val=X_val, t_val=t_val, d_val=d_val,
              verbose=False, patience=3, eval_every=5)

    # ------------------------------------------------------------------
    # Evaluation
    # ------------------------------------------------------------------
    test_nll = model.nll(X_te, t_te, d_te)
    print(f"-> Test NLL: {test_nll:.4f}\n")
    
    # Predict survival at the 3 specific horizon times
    S_test = model.predict_survival(X_te, horizons)  # Shape: (n_test, 3)
    
    for i, t_eval in enumerate(horizons):
        surv_probs = S_test[:, i]
        risk_scores = 1.0 - surv_probs
        
        # Calculate C-Index
        c_index, _, _, _, _ = concordance_index_ipcw(
            train_y, test_y, estimate=risk_scores, tau=t_eval
        )
        
        # Calculate Brier Score
        surv_probs_matrix = surv_probs.reshape(-1, 1) 
        _, brier_scores = brier_score(
            train_y, test_y, surv_probs_matrix, times=[t_eval]
        )
        
        print(f"Horizon t={t_eval:.2f}:")
        print(f"   C-Index (higher is better) : {c_index:.4f}")
        print(f"   Brier   (lower is better)  : {brier_scores[0]:.4f}")
    print("\n")

Fetching SUPPORT2 dataset...
n = 9105, censoring rate = 31.89%
Final feature matrix: (9105, 52)
Train 5463  Val 1821  Test 1821

Evaluation Horizons (25th, 50th, 75th quantiles): [ 14.  60. 258.]

Training Model -> alpha_mode: 'spline', rho: 0.0
-> Test NLL: 4.4357

Horizon t=14.00:
   C-Index (higher is better) : 0.8165
   Brier   (lower is better)  : 0.1102
Horizon t=60.00:
   C-Index (higher is better) : 0.7704
   Brier   (lower is better)  : 0.1680
Horizon t=258.00:
   C-Index (higher is better) : 0.7345
   Brier   (lower is better)  : 0.1904


Training Model -> alpha_mode: 'parametric', rho: 0.0
-> Test NLL: 4.4702

Horizon t=14.00:
   C-Index (higher is better) : 0.8097
   Brier   (lower is better)  : 0.1113
Horizon t=60.00:
   C-Index (higher is better) : 0.7650
   Brier   (lower is better)  : 0.1707
Horizon t=258.00:
   C-Index (higher is better) : 0.7322
   Brier   (lower is better)  : 0.1895


Training Model -> alpha_mode: 'parametric', rho: 1.0
-> Test NLL: 4.4278

Horizon t